# 🚨 UCF-Crime GCN Training — 14-Class Suspicious Activity Classifier
**Dataset:** [UCF Crime Dataset](https://www.kaggle.com/datasets/odins0n/ucf-crime-dataset)  
**Model:** Spatio-Temporal Graph Convolutional Network (ST-GCN)  
**Classes:** 14 (Abuse, Arrest, Arson, Assault, Burglary, Explosion, Fighting, NormalVideos, RoadAccidents, Robbery, Shooting, Shoplifting, Stealing, Vandalism)

### What this notebook does:
1. Installs dependencies
2. For each class folder, samples `.png` frames → runs YOLOv8-Pose → builds T=30 skeleton windows
3. Trains the ST-GCN (14-class) with per-epoch checkpointing
4. Saves best epoch weights + final results CSV + training plots


In [ ]:
# ── Cell 1: Install Dependencies ──
!pip install -q ultralytics
!pip install -q deep-sort-realtime
print("✅ Installations complete.")

In [ ]:
# ── Cell 2: Configuration — EDIT DATASET_ROOT IF NEEDED ──
import os, glob, json, csv, warnings, cv2, torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from collections import deque
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║  UPDATE THIS PATH — point it to the input dataset root.     ║
# ║  The folder should contain subfolders: Abuse/ Arrest/ etc.  ║
# ╚══════════════════════════════════════════════════════════════╝
DATASET_ROOT = "/kaggle/input/ucf-crime-dataset/UCF_Crimes/UCF_Crimes/Train"
TEST_ROOT    = "/kaggle/input/ucf-crime-dataset/UCF_Crimes/UCF_Crimes/Test"

# ── 14 UCF-Crime classes (index = class ID) ──
CLASSES = [
    "Abuse", "Arrest", "Arson", "Assault", "Burglary",
    "Explosion", "Fighting", "NormalVideos", "RoadAccidents",
    "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism"
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NORMAL_CLASS = "NormalVideos"

SEQUENCE_LEN     = 30      # frames per skeleton window
FRAMES_PER_CLASS = 600     # PNG frames sampled per class (increase for better accuracy)
BATCH_SIZE       = 32
EPOCHS           = 25
LEARNING_RATE    = 0.001
NUM_CLASSES      = len(CLASSES)

# ── Output paths ──
WORKING_DIR    = "/kaggle/working"
CKPT_DIR       = os.path.join(WORKING_DIR, "checkpoints")
BEST_WEIGHTS   = os.path.join(WORKING_DIR, "best_gcn_weights.pth")
RESULTS_CSV    = os.path.join(WORKING_DIR, "epoch_results.csv")
CACHE_X        = os.path.join(WORKING_DIR, "X_data.npy")
CACHE_Y        = os.path.join(WORKING_DIR, "Y_labels.npy")

os.makedirs(CKPT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"Classes: {NUM_CLASSES} → {CLASSES}")

In [ ]:
# ── Cell 3: Per-Frame Pose Extractor (YOLOv8-Pose) ──

class FramePoseExtractor:
    """
    Runs YOLOv8-Pose on individual PNG/JPG frames (no video needed).
    Returns a (17, 3) array of [x, y, confidence] per person.
    If no person detected → returns zeros.
    """
    def __init__(self, model_path="yolov8n-pose.pt"):
        print(f"Loading YOLOv8-pose model: {model_path} ...")
        self.model = YOLO(model_path)
        print("Model loaded.")

    def extract(self, img_path):
        try:
            frame = cv2.imread(img_path)
            if frame is None:
                return None
            # Upscale 64×64 Kaggle frames for better YOLO detection
            frame = cv2.resize(frame, (256, 256), interpolation=cv2.INTER_LINEAR)
            results = self.model(frame, classes=0, verbose=False)
            for result in results:
                if result.keypoints is None or result.keypoints.data is None:
                    continue
                if len(result.keypoints.data) == 0:
                    continue
                # Take person with highest detection confidence
                best_idx = 0
                if result.boxes is not None and len(result.boxes) > 1:
                    confs = result.boxes.conf.cpu().numpy()
                    best_idx = int(np.argmax(confs))
                kpts = result.keypoints.data[best_idx].cpu().numpy()  # (17, 3)
                return kpts.astype(np.float32)
        except Exception:
            pass
        return None


extractor = FramePoseExtractor("yolov8n-pose.pt")
print("✅ FramePoseExtractor ready.")

In [ ]:
# ── Cell 4: Skeleton Buffer + Preprocessor + Graph ──

class SkeletonBuffer:
    """Maintains a sliding window of SEQUENCE_LEN poses."""
    def __init__(self, max_frames=30):
        self.max_frames = max_frames
        self.buf = deque(maxlen=max_frames)

    def push(self, kpts):
        self.buf.append(kpts)

    def is_ready(self):
        return len(self.buf) > 0  # Returns sequence even if shorter (will be padded)

    def get_sequence(self):
        seq = list(self.buf)
        arr = np.array(seq, dtype=np.float32)   # (len, 17, 3)
        if len(seq) < self.max_frames:
            pad = np.tile(arr[0], (self.max_frames - len(seq), 1, 1))
            arr = np.concatenate((pad, arr), axis=0)
        return arr   # (T, 17, 3)


class GCNPreprocessor:
    """Normalises keypoints relative to hip centre; returns (1, 3, T, 17, 1) tensor."""
    def __init__(self, sequence_length=30):
        self.T = sequence_length
        self.V = 17

    def process(self, sequence):
        seq = sequence.copy()
        # Hip-centre normalisation
        hip_centre = (seq[:, 11, :2] + seq[:, 12, :2]) / 2.0
        seq[:, :, :2] -= hip_centre[:, np.newaxis, :]
        # Transpose to (C, T, V) then add N and M dims
        trans = np.transpose(seq, (2, 0, 1))                       # (3, T, 17)
        tensor = np.expand_dims(np.expand_dims(trans, 0), -1)      # (1, 3, T, 17, 1)
        return tensor.astype(np.float32)


class Graph:
    """COCO-17 joint graph: spatial strategy with 3 adjacency partitions."""
    def __init__(self):
        self.num_node    = 17
        self.max_hop     = 1
        self.center_node = 0
        self._build()

    def _build(self):
        self_link = [(i, i) for i in range(self.num_node)]
        neighbor_link = [
            (15,13),(13,11),(16,14),(14,12),(11,12),
            (5,11),(6,12),(5,6),(5,7),(7,9),(6,8),(8,10),
            (1,2),(0,1),(0,2),(1,3),(2,4),(0,5),(0,6)
        ]
        edges = self_link + neighbor_link
        A_raw = np.zeros((self.num_node, self.num_node))
        for i, j in edges:
            A_raw[i, j] = A_raw[j, i] = 1

        hop = np.full((self.num_node, self.num_node), np.inf)
        for d in range(self.max_hop + 1):
            reach = np.linalg.matrix_power(A_raw, d) > 0
            hop[reach] = np.minimum(hop[reach], d)

        valid = hop <= self.max_hop
        a_root = np.zeros_like(A_raw)
        a_cen  = np.zeros_like(A_raw)
        a_cef  = np.zeros_like(A_raw)
        for i in range(self.num_node):
            for j in range(self.num_node):
                if not valid[i, j]:
                    continue
                di = hop[i, self.center_node]
                dj = hop[j, self.center_node]
                if dj == di:  a_root[j, i] = 1
                elif dj < di: a_cen[j, i]  = 1
                else:         a_cef[j, i]  = 1

        def norm(m):
            s = m.sum(1, keepdims=True)
            s[s == 0] = 1
            return m / s

        self.A = np.stack([norm(a_root), norm(a_cen), norm(a_cef)])

preprocessor = GCNPreprocessor(sequence_length=SEQUENCE_LEN)
print("✅ Data processing classes ready.")

In [ ]:
# ── Cell 5: ST-GCN Model (14-class) ──

class SpatialGraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, K=3):
        super().__init__()
        self.K = K
        self.conv = nn.Conv2d(in_ch, out_ch * K, kernel_size=1)

    def forward(self, x, A):
        x = self.conv(x)
        N, KC, T, V = x.shape
        x = x.view(N, self.K, KC // self.K, T, V)
        x = torch.einsum('nkctv,kvw->nctw', x, A)
        return x.contiguous()


class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=9, stride=1):
        super().__init__()
        pad = ((kernel - 1) // 2, 0)
        self.conv = nn.Conv2d(in_ch, out_ch,
                              kernel_size=(kernel, 1),
                              stride=(stride, 1), padding=pad)
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)), inplace=True)


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, K=3, stride=1, dropout=0.5):
        super().__init__()
        self.sgcn    = SpatialGraphConv(in_ch, out_ch, K)
        self.tcn     = TCNBlock(out_ch, out_ch, stride=stride)
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU(inplace=True)
        if in_ch == out_ch and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=(stride, 1)),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x, A):
        res = self.residual(x)
        x   = self.drop(self.sgcn(x, A))
        x   = self.tcn(x)
        return self.relu(x + res)


class ActionRecognitionGCN(nn.Module):
    def __init__(self, num_classes=14, in_channels=3):
        super().__init__()
        graph = Graph()
        A = torch.tensor(graph.A, dtype=torch.float32)
        self.register_buffer('A', A)
        K  = A.size(0)
        V  = graph.num_node
        self.data_bn = nn.BatchNorm1d(in_channels * V)
        self.layers  = nn.ModuleList([
            STGCNBlock(in_channels, 64,  K, stride=1, dropout=0.3),
            STGCNBlock(64,  64,  K, stride=1, dropout=0.3),
            STGCNBlock(64,  128, K, stride=2, dropout=0.4),
            STGCNBlock(128, 128, K, stride=1, dropout=0.4),
            STGCNBlock(128, 256, K, stride=2, dropout=0.5),
            STGCNBlock(256, 256, K, stride=1, dropout=0.5),
        ])
        self.fc = nn.Linear(256, num_classes)
        self.V  = V
        self.C  = in_channels

    def forward(self, x):
        N, C, T, V, M = x.size()
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N * M, V, C, T).permute(0, 2, 3, 1).contiguous()
        for layer in self.layers:
            x = layer(x, self.A)
        x = F.avg_pool2d(x, x.size()[2:]).view(N, M, -1).mean(dim=1)
        return self.fc(x)

model = ActionRecognitionGCN(num_classes=NUM_CLASSES, in_channels=3).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ ActionRecognitionGCN ready  |  Parameters: {total_params:,}")

In [ ]:
# ── Cell 6: Build Dataset from PNG Frames ──
# Samples up to FRAMES_PER_CLASS images per class, builds sliding skeleton windows.
# Results are cached to .npy files — re-run skips extraction automatically.

def build_sequence_from_frames(img_paths, extractor, preprocessor, seq_len=30):
    """
    Takes a sequential list of frame paths (from ONE video/class),
    runs YOLO on each, fills a SkeletonBuffer, and returns a (1,3,T,17,1) tensor.
    Returns None if not enough valid keypoints were detected.
    """
    buf = SkeletonBuffer(max_frames=seq_len)
    zero_kpt = np.zeros((17, 3), dtype=np.float32)
    for img_path in img_paths:
        kpts = extractor.extract(img_path)
        buf.push(kpts if kpts is not None else zero_kpt)
    seq = buf.get_sequence()                   # (T, 17, 3)
    return preprocessor.process(seq)           # (1, 3, T, 17, 1)


if os.path.exists(CACHE_X) and os.path.exists(CACHE_Y):
    print("📂 Cache found — loading tensors from .npy files...")
    X_numpy = np.load(CACHE_X)
    Y_numpy = np.load(CACHE_Y)
    print(f"   X: {X_numpy.shape}  Y: {Y_numpy.shape}")
else:
    print("🔍 No cache — extracting skeletons from PNG frames (this may take 30-60 min)...")
    X_list, Y_list = [], []
    summary = {}

    for class_name in CLASSES:
        class_id  = CLASS_TO_IDX[class_name]
        class_dir = os.path.join(DATASET_ROOT, class_name)
        if not os.path.isdir(class_dir):
            print(f"  ⚠️  Folder not found: {class_dir}  — skipping.")
            continue

        # Gather all PNGs and sort to maintain temporal order
        all_imgs = sorted(glob.glob(os.path.join(class_dir, "*.png")) +
                          glob.glob(os.path.join(class_dir, "*.jpg")))
        if not all_imgs:
            print(f"  ⚠️  No images in {class_dir}  — skipping.")
            continue

        # Sub-sample to FRAMES_PER_CLASS images, evenly spaced
        step   = max(1, len(all_imgs) // FRAMES_PER_CLASS)
        frames = all_imgs[::step][:FRAMES_PER_CLASS]

        class_seqs = 0
        # Slide a window of seq_len over the sampled frames
        for start in range(0, len(frames) - SEQUENCE_LEN + 1, SEQUENCE_LEN // 2):
            window = frames[start: start + SEQUENCE_LEN]
            tensor = build_sequence_from_frames(window, extractor, preprocessor, SEQUENCE_LEN)
            if tensor is not None:
                X_list.append(tensor[0])    # (3, T, 17, 1)
                Y_list.append(class_id)
                class_seqs += 1

        summary[class_name] = class_seqs
        print(f"  [{class_id:02d}] {class_name:<20} → {class_seqs} sequences")

    if not X_list:
        raise RuntimeError("No sequences were built! Check DATASET_ROOT and folder names.")

    X_numpy = np.stack(X_list).astype(np.float32)
    Y_numpy = np.array(Y_list, dtype=np.int64)
    np.save(CACHE_X, X_numpy)
    np.save(CACHE_Y, Y_numpy)
    print(f"\n✅ Extraction done.  X: {X_numpy.shape}  Y: {Y_numpy.shape}")
    print(f"💾 Cached to {CACHE_X} and {CACHE_Y}")

# Class distribution
unique, counts = np.unique(Y_numpy, return_counts=True)
print("\n📊 Class distribution:")
for u, c in zip(unique, counts):
    tag = " ← Normal" if CLASSES[u] == NORMAL_CLASS else ""
    print(f"   [{u:02d}] {CLASSES[u]:<20} {c:5d} samples{tag}")

In [ ]:
# ── Cell 7: DataLoaders (balanced sampler for imbalanced classes) ──

X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
Y_tensor = torch.tensor(Y_numpy, dtype=torch.long)

dataset    = TensorDataset(X_tensor, Y_tensor)
n          = len(dataset)
train_size = int(0.80 * n)
val_size   = n - train_size
train_set, val_set = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Weighted sampler so rare classes are seen equally often during training
train_labels = Y_numpy[train_set.indices]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(float)
class_counts[class_counts == 0] = 1
weights      = 1.0 / class_counts[train_labels]
sampler      = WeightedRandomSampler(
    weights=torch.tensor(weights, dtype=torch.float64),
    num_samples=len(train_set),
    replacement=True
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {train_size} samples  |  Val: {val_size} samples")
print(f"Batch size: {BATCH_SIZE}  |  Epochs: {EPOCHS}")

In [ ]:
# ── Cell 8: Training Loop — Saves Checkpoint Every Epoch ──

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history        = []
best_val_acc   = -1.0
best_epoch     = -1

# CSV header
with open(RESULTS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "train_acc", "val_acc", "lr", "is_best"])

print(f"{'Epoch':>6}  {'Loss':>8}  {'Train%':>8}  {'Val%':>8}  {'Best':>6}")
print("─" * 50)

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────
    model.train()
    run_loss = correct = total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        run_loss += loss.item()
        preds     = logits.argmax(dim=1)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)

    train_loss = run_loss / len(train_loader)
    train_acc  = 100.0 * correct / total

    # ── Validate ───────────────────────────────────────────
    model.eval()
    v_correct = v_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            preds      = model(inputs).argmax(dim=1)
            v_correct += (preds == labels).sum().item()
            v_total   += labels.size(0)
    val_acc = 100.0 * v_correct / v_total if v_total > 0 else 0.0

    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    # ── Per-epoch checkpoint ────────────────────────────────
    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch:03d}_val{val_acc:.1f}.pth")
    torch.save({
        "epoch":      epoch,
        "model_state": model.state_dict(),
        "optimizer":  optimizer.state_dict(),
        "val_acc":    val_acc,
    }, ckpt_path)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_epoch   = epoch
        torch.save(model.state_dict(), BEST_WEIGHTS)
        star = "⭐"
    else:
        star = ""

    # ── Log ────────────────────────────────────────────────
    history.append({
        "epoch": epoch, "train_loss": train_loss,
        "train_acc": train_acc, "val_acc": val_acc, "lr": lr
    })
    with open(RESULTS_CSV, "a", newline="") as f:
        csv.writer(f).writerow([epoch, f"{train_loss:.4f}",
                                 f"{train_acc:.2f}", f"{val_acc:.2f}",
                                 f"{lr:.6f}", str(is_best)])

    print(f"{epoch:>6}  {train_loss:>8.4f}  {train_acc:>7.2f}%  {val_acc:>7.2f}%  {star}")

print("─" * 50)
print(f"\n🏆 Best Epoch : {best_epoch}  |  Best Val Accuracy : {best_val_acc:.2f}%")
print(f"💾 Best weights saved → {BEST_WEIGHTS}")
print(f"📋 Full results CSV  → {RESULTS_CSV}")
print(f"📁 All checkpoints   → {CKPT_DIR}/")

In [ ]:
# ── Cell 9: Training Curves ──
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

epochs_x   = [h["epoch"]      for h in history]
train_loss = [h["train_loss"] for h in history]
train_acc  = [h["train_acc"]  for h in history]
val_acc    = [h["val_acc"]    for h in history]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(epochs_x, train_loss, color="#e74c3c", lw=2, label="Train Loss")
axes[0].axvline(best_epoch, color="gold", ls="--", lw=1.5, label=f"Best Epoch ({best_epoch})")
axes[0].set_title("Training Loss per Epoch", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Accuracy
axes[1].plot(epochs_x, train_acc, color="#2ecc71", lw=2, label="Train Accuracy")
axes[1].plot(epochs_x, val_acc,   color="#3498db", lw=2, ls="--", label="Val Accuracy")
axes[1].axvline(best_epoch, color="gold", ls="--", lw=1.5, label=f"Best Epoch ({best_epoch})")
axes[1].set_title("Accuracy per Epoch", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plot_path = os.path.join(WORKING_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {plot_path}")

In [ ]:
# ── Cell 10: Per-Class Accuracy on Validation Set (Best Weights) ──

model.load_state_dict(torch.load(BEST_WEIGHTS, map_location=device))
model.eval()

class_correct = np.zeros(NUM_CLASSES)
class_total   = np.zeros(NUM_CLASSES)

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = model(inputs).argmax(dim=1)
        for p, l in zip(preds.cpu().numpy(), labels.cpu().numpy()):
            class_total[l]   += 1
            class_correct[l] += (p == l)

print(f"\n{'Class':<22} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("─" * 55)
for i, cname in enumerate(CLASSES):
    tot   = int(class_total[i])
    cor   = int(class_correct[i])
    acc   = 100.0 * cor / tot if tot > 0 else 0.0
    flag  = " ← Normal" if cname == NORMAL_CLASS else (" ⚠️ Suspicious" if acc < 50 else "")
    print(f"{cname:<22} {cor:>8} {tot:>8} {acc:>9.1f}%{flag}")

print("─" * 55)
overall = 100.0 * class_correct.sum() / class_total.sum()
print(f"{'OVERALL':<22} {int(class_correct.sum()):>8} {int(class_total.sum()):>8} {overall:>9.1f}%")

In [ ]:
# ── Cell 11: Sample Inference Demo (predict a single image window) ──

SUSPICIOUS_CLASSES = [c for c in CLASSES if c != NORMAL_CLASS]

def predict_activity(img_path_list, model, extractor, preprocessor, device):
    """
    img_path_list : list of 30 sequential frame paths from the SAME video.
    Returns: predicted class name, confidence %, and whether it is suspicious.
    """
    model.eval()
    from data_processing_helper import zero_kpt  # handled inline below
    zero_kpt = np.zeros((17, 3), dtype=np.float32)
    buf = SkeletonBuffer(max_frames=SEQUENCE_LEN)
    for img_path in img_path_list:
        kpts = extractor.extract(img_path)
        buf.push(kpts if kpts is not None else zero_kpt)
    seq    = buf.get_sequence()
    tensor = preprocessor.process(seq)
    inp    = torch.tensor(tensor, dtype=torch.float32).to(device)
    with torch.no_grad():
        probs = F.softmax(model(inp), dim=1)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    pred_cls = CLASSES[pred_idx]
    conf     = probs[pred_idx] * 100
    is_susp  = pred_cls != NORMAL_CLASS
    return pred_cls, conf, is_susp

# ── Quick demo using a random validation sample ──
sample_idx    = val_set.indices[0]
sample_input  = X_tensor[sample_idx].unsqueeze(0).to(device)
sample_label  = int(Y_tensor[sample_idx])

model.eval()
with torch.no_grad():
    probs    = F.softmax(model(sample_input), dim=1)[0].cpu().numpy()
pred_idx = int(np.argmax(probs))

print("\n🎬 Sample Prediction Demo")
print("─" * 45)
print(f"  Ground Truth  : {CLASSES[sample_label]}")
print(f"  Predicted     : {CLASSES[pred_idx]}  ({probs[pred_idx]*100:.1f}% confidence)")
is_susp = CLASSES[pred_idx] != NORMAL_CLASS
if is_susp:
    print(f"  🚨 SUSPICIOUS ACTIVITY DETECTED: {CLASSES[pred_idx].upper()}")
else:
    print(f"  ✅ Normal Activity — no alert.")

print("\n  Top-3 Predictions:")
top3 = np.argsort(probs)[::-1][:3]
for rank, idx in enumerate(top3, 1):
    s = "🚨" if CLASSES[idx] != NORMAL_CLASS else "✅"
    print(f"    {rank}. {s} {CLASSES[idx]:<20}  {probs[idx]*100:5.1f}%")

## Cell 12: Official UCF-Crime Test Set Evaluation
Extracts skeletons from `TEST_ROOT`, evaluates the best model, generates a confusion matrix and saves `test_results.csv`.

In [ ]:
# -- Cell 12: Official Test Set Evaluation (TEST_ROOT) --
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

TEST_CACHE_X       = os.path.join(WORKING_DIR, 'X_test.npy')
TEST_CACHE_Y       = os.path.join(WORKING_DIR, 'Y_test.npy')
TEST_RESULTS_CSV   = os.path.join(WORKING_DIR, 'test_results.csv')
TEST_FPC           = 200   # frames per class for test extraction
zero_kpt           = np.zeros((17, 3), dtype=np.float32)

# -- Extract or Load --
if os.path.exists(TEST_CACHE_X) and os.path.exists(TEST_CACHE_Y):
    print('Test cache found -- loading...')
    X_test_np = np.load(TEST_CACHE_X)
    Y_test_np = np.load(TEST_CACHE_Y)
else:
    print(f'Extracting skeletons from: {TEST_ROOT}')
    X_test_list, Y_test_list = [], []
    for class_name in CLASSES:
        class_id  = CLASS_TO_IDX[class_name]
        class_dir = os.path.join(TEST_ROOT, class_name)
        if not os.path.isdir(class_dir):
            print(f'  WARNING: {class_dir} not found -- skipping.')
            continue
        all_imgs = sorted(
            glob.glob(os.path.join(class_dir, '*.png')) +
            glob.glob(os.path.join(class_dir, '*.jpg'))
        )
        if not all_imgs:
            continue
        step   = max(1, len(all_imgs) // TEST_FPC)
        frames = all_imgs[::step][:TEST_FPC]
        seqs   = 0
        for start in range(0, len(frames) - SEQUENCE_LEN + 1, SEQUENCE_LEN // 2):
            window = frames[start: start + SEQUENCE_LEN]
            buf    = SkeletonBuffer(max_frames=SEQUENCE_LEN)
            for ip in window:
                kpts = extractor.extract(ip)
                buf.push(kpts if kpts is not None else zero_kpt)
            tensor = preprocessor.process(buf.get_sequence())
            X_test_list.append(tensor[0])
            Y_test_list.append(class_id)
            seqs += 1
        print(f'  [{class_id:02d}] {class_name:<20} -> {seqs} test sequences')
    X_test_np = np.stack(X_test_list).astype(np.float32)
    Y_test_np = np.array(Y_test_list, dtype=np.int64)
    np.save(TEST_CACHE_X, X_test_np)
    np.save(TEST_CACHE_Y, Y_test_np)
    print(f'Test tensors cached: {X_test_np.shape}')

print(f'Test samples: {len(X_test_np)}')

# -- DataLoader --
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test_np, dtype=torch.float32),
        torch.tensor(Y_test_np, dtype=torch.long)
    ),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

# -- Load best weights --
model.load_state_dict(torch.load(BEST_WEIGHTS, map_location=device))
model.eval()

# -- Inference --
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        preds = model(inputs.to(device)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# -- Per-class accuracy --
test_correct = np.zeros(NUM_CLASSES)
test_total   = np.zeros(NUM_CLASSES)
for p, l in zip(all_preds, all_labels):
    test_total[l]   += 1
    test_correct[l] += (p == l)
test_overall = 100.0 * test_correct.sum() / test_total.sum()

print('\n' + '='*60)
print(f'OFFICIAL TEST SET RESULTS  (Best Epoch: {best_epoch})')
print('='*60)
col_hdr = '{:<22} {:>8} {:>8} {:>8}'.format('Class','Correct','Total','Acc %')
print(col_hdr)
print('-'*55)
rows = []
for i, cname in enumerate(CLASSES):
    tot = int(test_total[i])
    cor = int(test_correct[i])
    acc = 100.0 * cor / tot if tot > 0 else 0.0
    tag = ' <- NORMAL' if cname == NORMAL_CLASS else (' [LOW]' if acc < 40 else '')
    print('{:<22} {:>8} {:>8} {:>7.1f}%  {}'.format(cname, cor, tot, acc, tag))
    rows.append([cname, cor, tot, f'{acc:.2f}'])
print('-'*55)
print('{:<22} {:>8} {:>8} {:>7.1f}%'.format('OVERALL',
      int(test_correct.sum()), int(test_total.sum()), test_overall))
print('='*60)

# -- Save test results CSV --
with open(TEST_RESULTS_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['class', 'correct', 'total', 'accuracy_pct'])
    w.writerows(rows)
    w.writerow(['OVERALL', int(test_correct.sum()), int(test_total.sum()), f'{test_overall:.2f}'])
print(f'Test CSV saved -> {TEST_RESULTS_CSV}')

# -- Confusion matrix --
cm      = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
            linewidths=0.5, vmin=0, vmax=1)
ax.set_title(f'Confusion Matrix -- Test Set (Best Epoch {best_epoch})', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
cm_path = os.path.join(WORKING_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved -> {cm_path}')


## Done -- Download Your Output Files

Go to the **Output** tab on the right panel and download:

| File | Description |
|------|-------------|
| `best_gcn_weights.pth` | Best model weights (highest val accuracy) |
| `epoch_results.csv` | Per-epoch train loss + val accuracy log |
| `test_results.csv` | Final per-class accuracy on UCF-Crime **Test** split |
| `training_curves.png` | Training loss + accuracy chart |
| `confusion_matrix.png` | Per-class confusion matrix heatmap (test set) |
| `checkpoints/` | One `.pth` checkpoint per epoch |

### Activate live inference locally:
1. Copy `best_gcn_weights.pth` into your `GNN_Project/` folder.
2. Update `inference.py`:
```python
model = ActionRecognitionGCN(num_classes=14, in_channels=3)
model.load_state_dict(torch.load('best_gcn_weights.pth'))
CLASS_LABELS = {
    0:'Abuse', 1:'Arrest', 2:'Arson', 3:'Assault', 4:'Burglary',
    5:'Explosion', 6:'Fighting', 7:'Normal', 8:'RoadAccidents',
    9:'Robbery', 10:'Shooting', 11:'Shoplifting', 12:'Stealing', 13:'Vandalism'
}
```
3. Run `python inference.py` -- your webcam now identifies exactly which crime class it is!
